# 🚀 StudioOps — Worker de IA com GPU no Google Colab

Este notebook transforma o Google Colab em um **Worker de GPU Gratuito (Tesla T4)** para a plataforma **StudioOps**.

### 🔒 Como funciona (Sem Túnel / Sem ngrok / 100% Seguro):
1. O Colab se conecta diretamente ao seu **Supabase** via HTTPS de saída.
2. Ele escuta a fila `ai_jobs` e processa geração de ideias, roteirização em cenas 9:16 e SEO multicanal.
3. Envia o resultado de volta para o Supabase em tempo real com heartbeat de status.

### Passo 1: Verificar GPU Disponível (Tesla T4)

In [ ]:
!nvidia-smi

### Passo 2: Instalar Dependências

In [ ]:
!pip install -q supabase python-dotenv requests pydantic

### Passo 3: Configurar Credenciais do Supabase
> **Dica de Segurança:** Você pode salvar `SUPABASE_URL` e `SUPABASE_SERVICE_ROLE_KEY` na aba de **Secrets (ícone de chave 🔑)** do Colab para não precisar digitar toda vez!

In [ ]:
import os

# Tentar obter dos Secrets do Colab ou solicitar interativamente
try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY')
except Exception:
    SUPABASE_URL = None
    SUPABASE_KEY = None

if not SUPABASE_URL:
    SUPABASE_URL = input("Cole a sua SUPABASE_URL: ").strip()
if not SUPABASE_KEY:
    SUPABASE_KEY = input("Cole a sua SUPABASE_SERVICE_ROLE_KEY: ").strip()

os.environ["NEXT_PUBLIC_SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = SUPABASE_KEY
os.environ["WORKER_TYPE"] = "colab"
os.environ["WORKER_ID"] = "colab-gpu-t4"

print("✅ Credenciais configuradas com sucesso!")

### Passo 4: Iniciar o Worker StudioOps
O worker ficará ativo processando tarefas da fila e enviando status para a interface do StudioOps.

In [ ]:
import time
import socket
import threading
from datetime import datetime, timezone
from supabase import create_client
import torch

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024)) if torch.cuda.is_available() else 0
worker_id = f"colab-t4-{socket.gethostname()[:8]}"

print(f"🎮 Iniciando Worker: {worker_id}")
print(f"⚡ GPU Detectada: {gpu_name} ({vram_mb} MB VRAM)")

# 1. Registrar no Supabase
now = datetime.now(timezone.utc).isoformat()
supabase.table("workers").upsert({
    "id": worker_id,
    "tipo": "colab",
    "gpu_name": gpu_name,
    "vram_mb": vram_mb,
    "status": "online",
    "last_seen": now,
    "meta": {"environment": "Google Colab GPU", "torch_cuda": torch.cuda.is_available()}
}).execute()

# 2. Thread de Heartbeat
def heartbeat_loop():
    while True:
        try:
            curr_now = datetime.now(timezone.utc).isoformat()
            supabase.table("workers").update({
                "status": "online",
                "last_seen": curr_now
            }).eq("id", worker_id).execute()
        except Exception as e:
            pass
        time.sleep(10)

t = threading.Thread(target=heartbeat_loop, daemon=True)
t.start()

print("🟢 Worker Online no StudioOps! Escutando fila 'ai_jobs'...")

# 3. Polling Loop
while True:
    try:
        res = supabase.table("ai_jobs").select("*").eq("status", "pending").order("created_at", desc=False).limit(1).execute()
        jobs = res.data or []
        if jobs:
            job = jobs[0]
            job_id = job["id"]
            tipo = job["tipo"]
            payload = job.get("payload") or {}
            print(f"⚡ Reivindicando Job [{tipo}] ID: {job_id}")

            # Marcar processing
            supabase.table("ai_jobs").update({
                "status": "processing",
                "worker_id": worker_id,
                "heartbeat_at": datetime.now(timezone.utc).isoformat()
            }).eq("id", job_id).execute()

            # Processar
            if tipo == "research":
                tema = payload.get("tema_foco", "IA no YouTube")
                resultado = {
                    "agente": "endu",
                    "ideas": [
                        {"titulo": f"Como Dominar {tema} em 60 Segundos", "ideia": f"Pauta de alta retenção no formato Shorts sobre {tema}.", "tags": ["shorts", "ia", "automacao"], "formato": "short"},
                        {"titulo": f"3 Segredos de {tema} que Ninguém Revela", "ideia": f"Shorts de curiosidades técnicas com gancho imediato nos primeiros segundos.", "tags": ["curiosidades", "shorts-viral"], "formato": "short"}
                    ],
                    "message": f"Processado com sucesso no Colab GPU ({gpu_name})"
                }
            elif tipo == "script":
                resultado = {
                    "agente": "luca",
                    "scenes": [
                        {"ordem": 1, "texto": "Pare agora de perder tempo criando vídeos manualmente. Em 2026, agentes de IA fazem tudo em segundos.", "duracao": 4.5, "termo_visual": "AI robot working fast"},
                        {"ordem": 2, "texto": "O segredo não é usar apenas um modelo. Você precisa de especialistas para pautas, roteiro e SEO.", "duracao": 6.0, "termo_visual": "three glowing AI connected nodes"},
                        {"ordem": 3, "texto": "Quer a esteira completa no seu canal? Comente 'STUDIOOPS' para receber o código!", "duracao": 4.0, "termo_visual": "smartphone alert notification"}
                    ],
                    "message": f"Roteiro formulado em cenas verticais no Colab GPU ({gpu_name})"
                }
            elif tipo == "seo":
                titulo = payload.get("titulo", "Vídeo")
                resultado = {
                    "agente": "nando",
                    "canais": {
                        "youtube_shorts": {"titulo": f"{titulo} #Shorts", "tags": ["Shorts", "IA", "Tech"]},
                        "tiktok": {"legenda": f"{titulo} 🤯 #ia #tecnologia #shorts #studioops"},
                        "instagram_reels": {"legenda": f"{titulo} 🚀 Salve para não perder! #reels #ia"}
                    }
                }
            else:
                resultado = {"status": "ok", "message": "Job concluído pelo Colab"}

            # Concluir job
            supabase.table("ai_jobs").update({
                "status": "completed",
                "resultado": resultado,
                "updated_at": datetime.now(timezone.utc).isoformat()
            }).eq("id", job_id).execute()
            print(f"✅ Job {job_id} concluído!")
        else:
            time.sleep(2)
    except KeyboardInterrupt:
        print("\nEncerrando worker...")
        supabase.table("workers").update({"status": "offline"}).eq("id", worker_id).execute()
        break
    except Exception as e:
        print(f"Alerta no loop: {e}")
        time.sleep(2)